In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
#
from src.data.preprocess import preprocess_text

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\RGUKT\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\RGUKT\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
train_df = pd.read_csv(
    "../data/interim/train_cleaned.csv"
)

test_df = pd.read_csv(
    "../data/interim/test_cleaned.csv"
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (120000, 6)
Test: (7600, 5)


In [4]:
#Inspect original text
train_df[["title", "description", "text"]].head()

,title,description,text
0,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


In [5]:
sample = train_df.loc[0, "text"]

print(sample)

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [6]:
processed = preprocess_text(sample)

print(processed)

wall st bear claw back black reuters reuters shortsellers wall street dwindlingband ultracynics seeing green


In [7]:
#now apply the preprocess to the entire text
train_df["clean_text"] = train_df["text"].apply(
    preprocess_text
)

In [8]:
test_df["clean_text"] = test_df["text"].apply(
    preprocess_text
)

In [9]:
train_df[
    ["text", "clean_text"]
].head()

,text,clean_text
0,Wall St. Bears Claw Back Into the Black (Reute...,wall st bear claw back black reuters reuters s...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle look toward commercial aerospace reute...
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil economy cloud stock outlook reuters reuter...
3,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halt oil export main southern pipeline re...
4,"Oil prices soar to all-time record, posing new...",oil price soar alltime record posing new menac...


In [10]:
#Check for empty documents
empty_train = (
    train_df["clean_text"].str.strip() == ""
).sum()

empty_test = (
    test_df["clean_text"].str.strip() == ""
).sum()

print("Empty train documents:", empty_train)
print("Empty test documents:", empty_test)

Empty train documents: 0
Empty test documents: 0


In [11]:
#
train_df = train_df[
    train_df["clean_text"].str.strip() != ""
].copy()

test_df = test_df[
    test_df["clean_text"].str.strip() != ""
].copy()

In [12]:
#10. Analyze preprocessing impact
train_df["original_word_count"] = (
    train_df["text"]
    .str.split()
    .str.len()
)

train_df["clean_word_count"] = (
    train_df["clean_text"]
    .str.split()
    .str.len()
)

In [13]:
train_df[
    [
        "original_word_count",
        "clean_word_count"
    ]
].describe()

,original_word_count,clean_word_count
count,120000.000000,120000.000000
mean,37.844617,24.874800
std,10.088702,6.373566
min,4.000000,3.000000
25%,32.000000,21.000000
50%,37.000000,25.000000
75%,43.000000,28.000000
max,177.000000,100.000000


In [14]:
#11. Save the processed dataset
processed_dir = Path("../data/processed")

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)